# iZM564 模型构建与 Biolog 底物利用测试

> 从 KEGG 注释的 Excel 文件构建运动发酵单胞菌（*Zymomonas mobilis*）iZM564 代谢模型，
> 依次完成 FBA / pFBA 分析、模型一致性检查，并用 pFBA 模拟 Biolog 表型芯片，
> 逐一测试碳 / 氮 / 磷 / 硫源是否可被利用。

## 目录

- [1. 环境与配置](#sec-1)
- [2. 模型构建（Excel → cobra 模型）](#sec-2)
  - [2.1 创建空模型与加载数据](#sec-2-1)
  - [2.2 添加代谢物](#sec-2-2)
  - [2.3 添加反应](#sec-2-3)
  - [2.4 添加基因注释](#sec-2-4)
  - [2.5 模型规模检查](#sec-2-5)
- [3. FBA 分析](#sec-3)
  - [3.1 目标函数与环境条件](#sec-3-1)
  - [3.2 pFBA 通量分布](#sec-3-2)
- [4. 模型一致性检查](#sec-4)
- [5. 模型输出（SBML / JSON）](#sec-5)
- [6. Biolog 底物利用测试](#sec-6)
  - [6.1 通用工具函数](#sec-6-1)
  - [6.2 碳源测试（C）](#sec-6-2)
  - [6.3 氮源测试（N）](#sec-6-3)
  - [6.4 磷源测试（P）](#sec-6-4)
  - [6.5 硫源测试（S）](#sec-6-5)
- [7. 附录](#sec-7)

## 运行说明

按单元格顺序从上到下依次运行。第 2~5 节构建并导出模型；
第 6 节 Biolog 测试相互独立，每次从 `model/` 目录的 JSON 重新加载模型。

## <a id="sec-1"></a> 1. 环境与配置

**依赖声明**（下表为开发时验证过的版本）：

| 包 | 用途 | 版本 |
| --- | --- | --- |
| cobra | 代谢建模与 FBA | 0.22.1 |
| pandas | 读取 Excel 工作表 | 1.5.3 |
| numpy | 数值计算 / 随机种子 | 1.23.5 |
| gurobipy | 线性规划求解器（推荐） | 12.0.3 |
| xlrd | 读取 .xls 格式 Excel | 2.0.1 |

> 求解器说明：Gurobi 数值上比 GLPK 稳定，同一模型可得到一致结果；
> 若未安装 Gurobi，cobra 会自动回退到其他求解器，数值结果可能略有差异。

In [1]:
'''1.1 依赖导入与环境检查'''
import ast
import importlib.metadata
from pathlib import Path

import numpy as np
import pandas as pd
import cobra
from cobra import Model, Metabolite, Reaction

# ---- 固定随机种子 ----
# 本流程本身不含随机过程；固定种子可保证未来加入随机采样步骤时结果可复现
np.random.seed(42)

# ---- 求解器配置 ----
# 使用 Gurobi：数值稳定、结果可复现（GLPK 对部分模型会给出不一致结果）
cobra_config = cobra.Configuration()
try:
    cobra_config.solver = "gurobi"
except Exception as exc:  # gurobi 未安装时回退到默认求解器
    print(f"警告：无法启用 Gurobi（{exc}），回退到默认求解器 {cobra_config.solver}")


# ---- 依赖版本检查 ----
def show_versions():
    """打印关键依赖的版本号，便于复现运行环境。"""
    for pkg in ("cobra", "pandas", "numpy", "gurobipy", "xlrd"):
        try:
            version = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            version = "未安装"
        print(f"{pkg:>10s} : {version}")


show_versions()
print("当前求解器:", cobra_config.solver)

     cobra : 0.22.1
    pandas : 1.5.3
     numpy : 1.23.5
  gurobipy : 12.0.3
      xlrd : 2.0.1
当前求解器: <module 'optlang.gurobi_interface' from 'd:\\Python\\Python39\\lib\\site-packages\\optlang\\gurobi_interface.py'>


In [2]:
'''1.2 全局配置：路径、模型命名与通量边界
修改本单元格即可将整个流程迁移到其他目录 / 数据文件。'''

# ---- 路径配置 ----
DATA_DIR = Path(".")
MODEL_DIR = Path(".") / "model"
EXCEL_FILE = DATA_DIR / "iZM564_20260807_KEGG_met.xls"  # 模型源数据（KEGG 注释）

# ---- 模型命名 ----
MODEL_NAME = "iZM564_20260807_KEGG_met"  # 模型 ID，须与 Excel 文件名前缀一致

# ---- 通量边界常量（单位：mmol/gDW/h）----
EXCHANGE_LIMIT = 1000  # 交换反应上界：表示“充分供应”
GLC_UPTAKE = 10        # FBA 中葡萄糖摄入速率（绝对值）
BIOMASS_UB = 1000      # 生物量反应上界（放开限制）

# 确保输出目录存在
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## <a id="sec-2"></a> 2. 模型构建（Excel → cobra 模型）

流程：新建空模型 → 依次添加代谢物、反应、基因注释。
三张工作表均来自 KEGG 注释的 Excel 文件，`id` 列与模型内 ID 一一对应。

### <a id="sec-2-1"></a> 2.1 创建空模型与加载数据

In [4]:
# 创建空模型（模型 ID 与 Excel 文件名保持一致）
model = Model(MODEL_NAME)

# 读取三张工作表（header=0 表示第一行为列名）
sheet_names = {"metabolites", "reactions", "genes"}
sheets = {
    name: pd.read_excel(EXCEL_FILE, sheet_name=name, header=0)
    for name in sheet_names
}
met_sheet = sheets["metabolites"]
rxn_sheet = sheets["reactions"]
gene_sheet = sheets["genes"]

print(f"代谢物表 {len(met_sheet)} 行，反应表 {len(rxn_sheet)} 行，基因表 {len(gene_sheet)} 行")

代谢物表 1641 行，反应表 1670 行，基因表 2026 行


### <a id="sec-2-2"></a> 2.2 添加代谢物

Excel 中 `formula` 列为字符串，值为 `'none'` 表示未知分子式（不设置该属性）；
`annotation` 列以字符串化 dict 存储 KEGG / SBO 等注释，用 `ast.literal_eval` 安全解析（替代裸 `eval`）。

In [5]:
def add_metabolites_from_sheet(model, sheet):
    """将代谢物工作表逐行加入模型。

    参数
    ----
    model : cobra.Model
        目标模型（当前应为空模型）。
    sheet : pandas.DataFrame
        "metabolites" 工作表，必需列：id, name, formula, compartment, charge, annotation。

    返回
    ----
    int
        添加的代谢物数量。
    """
    for _, row in sheet.iterrows():
        # formula 为 'none'（或缺失）时表示未知分子式，不设置该属性
        formula = row["formula"]
        kwargs = {}
        if not (pd.isna(formula) or str(formula) == "none"):
            kwargs["formula"] = str(formula)

        met = Metabolite(
            str(row["id"]),
            name=str(row["name"]),
            compartment=str(row["compartment"]),
            charge=row["charge"],
            **kwargs,
        )
        model.add_metabolites([met])
        # 写入数据库注释（字符串化 dict -> dict）
        met.annotation = ast.literal_eval(row["annotation"])
    return len(sheet)


add_metabolites_from_sheet(model, met_sheet)
print("代谢物添加完成")

代谢物添加完成


### <a id="sec-2-3"></a> 2.3 添加反应

- 化学计量从 `reaction_eq_annotation` 读取，格式为 `{"<代谢物ID>": <系数>, ...}`，
  如 `{"cpd01221_p0": -1.0, "cpd01221_e0": 1.0}`。
- `GPR` 列为空（NaN）时表示该反应无基因规则，直接跳过。

In [6]:
def add_reactions_from_sheet(model, sheet):
    """将反应工作表逐行加入模型。

    参数
    ----
    model : cobra.Model
        已含代谢物的模型。
    sheet : pandas.DataFrame
        "reactions" 工作表，必需列：id, name, lower_bound, upper_bound,
        annotation, GPR, reaction_eq_annotation。

    返回
    ----
    int
        添加的反应数量。
    """
    for _, row in sheet.iterrows():
        rxn = Reaction(str(row["id"]))
        rxn.name = str(row["name"])
        rxn.lower_bound = float(row["lower_bound"])
        rxn.upper_bound = float(row["upper_bound"])
        rxn.annotation = ast.literal_eval(row["annotation"])
        # 仅当 GPR 为字符串时才设置基因-蛋白质-反应规则（NaN 表示无规则）
        if isinstance(row["GPR"], str):
            rxn.gene_reaction_rule = row["GPR"]
        model.add_reactions([rxn])
        # 写入化学计量：{"代谢物ID": 系数, ...}
        rxn.add_metabolites(ast.literal_eval(row["reaction_eq_annotation"]))
    return len(sheet)


add_reactions_from_sheet(model, rxn_sheet)
print("反应添加完成")

反应添加完成


### <a id="sec-2-4"></a> 2.4 添加基因注释

从基因表构建 `gene_id → {gene_name, annotation}` 映射，再按 ID 回填到模型中的基因；
`annotation` 中会追加 `product`（基因产物描述）字段。

In [7]:
def annotate_genes_from_sheet(model, sheet):
    """为模型中的每个基因写入名称与注释。

    参数
    ----
    model : cobra.Model
        已含反应（进而含基因）的模型。
    sheet : pandas.DataFrame
        "genes" 工作表，必需列：Gene_id, Gene_name, annotation, product。

    返回
    ----
    int
        映射表中基因条目数。
    """
    # 构建 gene_id -> 基因信息 的映射
    # （原代码 else 分支中误用了未定义变量，此处统一先解析 annotation，已修复）
    gene_info = {}
    for _, row in sheet.iterrows():
        annotation = ast.literal_eval(row["annotation"])
        annotation["product"] = str(row["product"])  # 追加产物描述
        gene_info[str(row["Gene_id"])] = {
            "gene_name": str(row["Gene_name"]),
            "annotation": annotation,
        }

    # 按 ID 回填到模型中已存在的基因
    for gene in model.genes:
        info = gene_info[gene.id]
        gene.annotation = info["annotation"]
        gene.name = info["gene_name"]
    return len(gene_info)


annotate_genes_from_sheet(model, gene_sheet)
print("基因注释完成")

基因注释完成


### <a id="sec-2-5"></a> 2.5 模型规模检查

In [8]:
print(f"{len(model.reactions)} reactions")
print(f"{len(model.metabolites)} metabolites")
print(f"{len(model.genes)} genes")

1670 reactions
1641 metabolites
564 genes


## <a id="sec-3"></a> 3. FBA 分析

FBA（通量平衡分析）在稳态假设下最大化目标通量（此处为生物量）。

### <a id="sec-3-1"></a> 3.1 目标函数与环境条件

- 目标：最大化生物量反应 `ZM_BIOMASS`；
- 条件：葡萄糖摄入 `-10 mmol/gDW/h` 为唯一碳源，木糖与 O₂ 均不供应（**无氧**）。

In [9]:
# ---- 目标函数 ----
model.objective = "ZM_BIOMASS"
model.reactions.get_by_id("ZM_BIOMASS").upper_bound = BIOMASS_UB

# ---- 底物供应（摄入方向为负通量）----
model.reactions.get_by_id("EX_glc_D_e").lower_bound = -GLC_UPTAKE  # 葡萄糖
model.reactions.get_by_id("EX_xyl_D_e").lower_bound = 0            # 木糖不供应
model.reactions.get_by_id("EX_o2_e").lower_bound = 0               # 无氧

# ---- FBA 求解并输出汇总 ----
fba_solution = model.optimize()
print("生物量 (FBA):", fba_solution.objective_value)
print()
print(model.summary().to_string(names=True))

生物量 (FBA): 0.04681405214109416

Objective
1.0 Biomass reaction = 0.04681405214109416

Uptake
------
Metabolite    Reaction      Flux  C-Number  C-Flux
      Ca2+    EX_ca2_e 2.611E-05         0   0.00%
       Cl-     EX_cl_e 2.952E-05         0   0.00%
      Fe+2    EX_fe2_e 1.874E-05         0   0.00%
 D-Glucose  EX_glc_D_e        10         6 100.00%
        K+      EX_k_e 2.676E-05         0   0.00%
        Mg    EX_mg2_e 4.305E-05         0   0.00%
 Molybdate   EX_mobd_e 6.345E-06         0   0.00%
       nh4    EX_nh4_e     0.502         0   0.00%
 Phosphate     EX_pi_e   0.07058         0   0.00%
       PAN EX_pnto_R_e 1.369E-06         9   0.00%
   Sulfate    EX_so4_e  0.005156         0   0.00%
   Thiamin    EX_thm_e 2.472E-06        12   0.00%

Secretion
---------
    Metabolite   Reaction       Flux  C-Number C-Flux
       Acetate    EX_ac_e   -0.02586         2  0.09%
           CO2   EX_co2_e    -0.0183         1  0.03%
  Deoxyuridine  EX_duri_e   -0.02056         9  0.32%


### <a id="sec-3-2"></a> 3.2 pFBA 通量分布

pFBA（通量变异性最小化）在最大生物量下进一步最小化总通量，
使通量分布唯一且更接近实验观测。此处展示 |通量| > 1e-4 的反应。

In [10]:
pfba_solution = cobra.flux_analysis.pfba(model)
print("生物量 (pFBA):", pfba_solution["ZM_BIOMASS"])
print()

# 展示 |通量| > 1e-4 的所有活跃反应
active_fluxes = pfba_solution.fluxes[abs(pfba_solution.fluxes) > 1e-4]
for rxn_id, flux in active_fluxes.items():
    rxn_str = model.reactions.get_by_id(rxn_id).build_reaction_string(
        use_metabolite_names=True)
    print(f"{rxn_id}\t{flux:.4f}\t{rxn_str}")

生物量 (pFBA): 0.04681405214109416

ATPM	3.7100	H2O + ATP --> ADP + Phosphate + H+
EX_ac_e	0.0259	Acetate --> 
EX_co2_e	0.0183	CO2 --> 
EX_duri_e	0.0206	Deoxyuridine --> 
EX_fald_e	0.0597	Formaldehyde --> 
EX_for_e	0.0839	Formate --> 
EX_glc_D_e	-10.0000	D-Glucose <=> 
EX_h_e	18.5363	H+ <=> 
EX_h2o_e	2.2468	H2O <=> 
EX_lac_D_e	19.2465	D-Lactate --> 
EX_nh4_e	-0.5020	nh4 <=> 
EX_pi_e	-0.0706	Phosphate <=> 
EX_so4_e	-0.0052	Sulfate <=> 
EX_succ_e	0.0295	Succinate --> 
HPN	0.0017	H2O + Bacteriohopanetetrol-acetylglucosamine --> Acetate + H+ + Tetrahydroxybacteriohopane-glucosamine
HPN4	0.0017	UDP-glucosamine + Tetrahydroxybacteriohopane --> UDP + H+ + Bacteriohopanetetrol-acetylglucosamine
rxn00001	0.1234	H2O + PPi --> 2.0 Phosphate + H+
rxn00011	0.0490	Pyruvate + TPP + H+ --> CO2 + 2-Hydroxyethyl-ThPP
rxn00085	-0.4149	NADP + 2.0 L-Glutamate <-- NADPH + 2-Oxoglutarate + L-Glutamine + H+
rxn00097	0.0960	ATP + AMP + H+ <=> 2.0 ADP
rxn00102	-0.1216	H+ + H2CO3 <=> H2O + CO2
rxn00119	0.0319	ATP +

## <a id="sec-4"></a> 4. 模型一致性检查

逐一检查反应的质量守恒与电荷守恒（`check_mass_balance` 返回非空 dict 表示不平衡）。
交换反应（`EX_*`）为伪反应，默认不检查。

> 说明：KEGG 来源的模型中，少量不平衡反应属于正常现象（如生物量反应、
> 需额外质子平衡的反应），可结合 SBO 注释人工确认：
> `SBO:0000627` 交换 / `SBO:0000628` 需求 / `SBO:0000629` 生物量
> / `SBO:0000631` 伪反应 / `SBO:0000632` 库反应。

In [11]:
def check_mass_balance(model, verbose=True):
    """统计并展示质量 / 电荷不平衡的反应（排除交换反应 EX_*）。

    参数
    ----
    model : cobra.Model
    verbose : bool
        为 True 时逐条打印不平衡反应的详细信息。

    返回
    ----
    list[str]
        不平衡反应 ID 列表。
    """
    unbalanced = []
    for rxn in model.reactions:
        if "EX_" in rxn.id:
            continue
        imbalance = rxn.check_mass_balance()
        if imbalance:  # 非空 dict 表示质量或电荷不平衡
            unbalanced.append(rxn.id)
            if verbose:
                print(rxn.id, imbalance)
                print(rxn.build_reaction_string(use_metabolite_names=True))
                for met in rxn.metabolites:
                    print(f"{met.id}\t{met.name}\t{met.formula}\t{met.charge}")
                print("-" * 26)
    print(f"unbalanced reactions: {len(unbalanced)}")
    return unbalanced


unbalanced_rxns = check_mass_balance(model)

HPN4 {'C': 2.0, 'H': 2.0, 'O': 1.0}
UDP-glucosamine + Tetrahydroxybacteriohopane --> UDP + H+ + Bacteriohopanetetrol-acetylglucosamine
cpd01482_c0	UDP-glucosamine	C15H24N3O16P2	-1
cpd33171_c0	Tetrahydroxybacteriohopane	C35H62O4	0
thbhaga_c0	Bacteriohopanetetrol-acetylglucosamine	C43H75NO9	0
cpd00014_c0	UDP	C9H12N2O12P2	-2
cpd00067_c0	H+	H	1
--------------------------
rxn14057 {'charge': -7, 'H': -1}
3 H2O + H2S + 6 Oxidizedferredoxin <-- 6 H+ + Sulfite + 6 Reducedferredoxin
cpd00001_c0	H2O	H2O	0
cpd00239_c0	H2S	H2S	0
cpd11621_c0	Oxidizedferredoxin	Fe2R4S6	6
cpd00067_c0	H+	H	1
cpd00081_c0	Sulfite	HO3S	-1
cpd11620_c0	Reducedferredoxin	Fe2R4S6	4
--------------------------
rxn43433 {'charge': -2.0}
S-Adenosyl-L-methionine + H+ + Diploptene --> L-Methionine + Adenosylhopane
cpd03751_c0	Diploptene	C30H50	0
cpd00017_c0	S-Adenosyl-L-methionine	C15H23N6O5S	1
cpd00067_c0	H+	H	1
cpd31750_c0	Adenosylhopane	C40H63N5O3	0
cpd00060_c0	L-Methionine	C5H11NO2S	0
--------------------------
ZM_BIOMASS {'H'

## <a id="sec-5"></a> 5. 模型输出（SBML / JSON）

将构建好的模型导出为 SBML 与 JSON，供下游模块（eciZM564 构建、ETGEM 校正等）加载。

In [12]:
# sbml_path = MODEL_DIR / f"{MODEL_NAME}.sbml"
json_path = MODEL_DIR / f"{MODEL_NAME}.json"

# cobra.io.write_sbml_model(model, str(sbml_path))
cobra.io.save_json_model(model, str(json_path))

print("已导出：")
# print(" ", sbml_path)
print(" ", json_path)

已导出：
  model\iZM564_20260807_KEGG_met.json


## <a id="sec-6"></a> 6. Biolog 底物利用测试

**测试原理**：对每个候选代谢物，从 `model/` 下导出的 JSON 重新加载模型，
屏蔽默认供应源（碳源测试屏蔽葡萄糖、氮源屏蔽 NH₄⁺、磷源屏蔽磷酸、硫源屏蔽硫酸根），
仅开放该候选物的交换反应，再以 pFBA 求解生物量；生物量 > 1e-6 即认为该底物可利用。

> 金属离子交换反应保持无限制；每次测试均重新加载模型，保证测试间相互独立。

### <a id="sec-6-1"></a> 6.1 通用工具函数

In [12]:
def add_exchange_rxn(model, met_id, unlimited=True):
    """为代谢物 met_id 添加交换反应 EX_<met_id>。

    参数
    ----
    model : cobra.Model
    met_id : str
        代谢物 ID（含隔间后缀，如 'cpd00020_e0'）。
    unlimited : bool
        True  → 边界 (-1000, 1000)，允许任意摄入 / 分泌；
        False → 边界 (0, 1000)，仅允许分泌。

    返回
    ----
    cobra.Model
        添加交换反应后的模型。
    """
    met = model.metabolites.get_by_id(met_id)
    rxn = Reaction(f"EX_{met_id}")
    rxn.add_metabolites({met: -1})
    rxn.bounds = (-EXCHANGE_LIMIT, EXCHANGE_LIMIT) if unlimited else (0, EXCHANGE_LIMIT)
    model.add_reaction(rxn)
    return model


def run_biolog_test(candidates, compartment, blocked_exchange,
                    set_h_exchange=False, no_uptake_exceptions=(),
                    show_errors=False, model_dir=None, model_name=None):
    """对一组候选代谢物逐一执行 Biolog 底物利用测试。

    对每个候选物：从 JSON 重新加载模型 → 添加该物的交换反应 →
    屏蔽默认供应源 → pFBA 求解生物量。

    参数
    ----
    candidates : list[str]
        候选代谢物 ID 列表（不含隔间后缀）。
    compartment : str
        隔间后缀，如 '_e0'（胞外）或 '_c0'（胞质）。
    blocked_exchange : str
        需要屏蔽的默认交换反应 ID，如 'EX_glc_D_e'。
    set_h_exchange : bool
        是否将质子交换反应 EX_h_e 上界设为 EXCHANGE_LIMIT。
    no_uptake_exceptions : tuple[str, ...]
        仅允许分泌（unlimited=False）的代谢物 ID。
    show_errors : bool
        出错时是否打印详细错误信息（False 时仅打印 0）。
    model_dir : Path
        模型 JSON 所在目录，默认取全局 MODEL_DIR。
    model_name : str
        模型基础文件名，默认取全局 MODEL_NAME。

    返回
    ----
    dict[str, float]
        代谢物 ID → pFBA 生物量（不可利用或求解失败时为 0）。
    """
    model_dir = model_dir or MODEL_DIR
    model_name = model_name or MODEL_NAME
    base_json = model_dir / f"{model_name}.json"

    results = {}
    for met_id in candidates:
        # 每次重新加载模型，保证各候选物测试相互独立
        model = cobra.io.load_json_model(str(base_json))
        full_met_id = f"{met_id}{compartment}"
        exchange_id = f"EX_{full_met_id}"
        try:
            add_exchange_rxn(model, full_met_id,
                             unlimited=met_id not in no_uptake_exceptions)
            model.medium[exchange_id] = EXCHANGE_LIMIT
            if set_h_exchange:
                model.medium["EX_h_e"] = EXCHANGE_LIMIT
            # 屏蔽默认供应源
            model.reactions.get_by_id(blocked_exchange).bounds = (0, 0)
            model.medium[blocked_exchange] = 0

            biomass = cobra.flux_analysis.pfba(model)["ZM_BIOMASS"]
            results[met_id] = biomass if biomass > 1e-6 else 0.0
            print(biomass if biomass > 1e-6 else 0)
        except Exception as exc:
            results[met_id] = 0.0
            print(f"0\t{exc}" if show_errors else 0)
    return results

### <a id="sec-6-2"></a> 6.2 碳源测试（C）

候选碳源位于胞外（`_e0`）。屏蔽葡萄糖后逐一测试；
出错时打印详细错误信息（便于排查缺失的转运反应）。

In [13]:
# Biolog 碳源候选列表（来源：Biolog 表型芯片碳源，含少量重复项，保留以对齐原始结果）
C_CANDIDATES = [
    'cpd00020', 'cpd00024', 'cpd00027', 'cpd00029', 'cpd00033', 'cpd00036', 'cpd00040', 'cpd00047', 'cpd00064', 'cpd00072', 'cpd00076', 'cpd00079',
    'cpd00080', 'cpd00089', 'cpd00094', 'cpd00100', 'cpd00106', 'cpd00117', 'cpd00118', 'cpd00122', 'cpd00130', 'cpd00136', 'cpd00137', 'cpd00138',
    'cpd00139', 'cpd00141', 'cpd00142', 'cpd00154', 'cpd00155', 'cpd00157', 'cpd00158', 'cpd00159', 'cpd00161', 'cpd00162', 'cpd00180', 'cpd00182',
    'cpd00184', 'cpd00185', 'cpd00208', 'cpd00211', 'cpd00222', 'cpd00224', 'cpd00227', 'cpd00246', 'cpd00248', 'cpd00249', 'cpd00023', 'cpd00053',
    'cpd00276', 'cpd00280', 'cpd00281', 'cpd00306', 'cpd00308', 'cpd00314', 'cpd00320', 'cpd00322', 'cpd00339', 'cpd11685', 'cpd00366', 'cpd00374',
    'cpd19003', 'cpd00380', 'cpd00382', 'cpd00386', 'cpd00386', 'cpd00392', 'cpd00396', 'cpd00417', 'cpd00432', 'cpd00438', 'cpd00453', 'cpd00477',
    'cpd11717', 'cpd00489', 'cpd27563', 'cpd11601', 'cpd00054', 'cpd11594', 'cpd00550', 'cpd00551', 'cpd00119', 'cpd11748', 'cpd00588', 'cpd00589',
    'cpd00599', 'cpd00607', 'cpd00571', 'cpd00611', 'cpd00652', 'cpd00666', 'cpd00108', 'cpd00728', 'cpd00747', 'cpd00750', 'cpd00751', 'cpd00781',
    'cpd00794', 'cpd00797', 'cpd00820', 'cpd00832', 'cpd00035', 'cpd01030', 'cpd01055', 'cpd11879', 'cpd01067', 'cpd01107', 'cpd01113', 'cpd01133',
    'cpd01171', 'cpd00060', 'cpd01200', 'cpd00361', 'cpd01242', 'cpd01262', 'cpd01293', 'cpd01307', 'cpd00179', 'cpd01363', 'cpd00066', 'cpd19018',
    'cpd01502', 'cpd00051', 'cpd01735', 'cpd01947', 'cpd11602', 'cpd02143', 'cpd02274', 'cpd02351', 'cpd02599', 'cpd03161', 'cpd03198', 'cpd03320',
    'cpd03561', 'cpd03696', 'cpd03734', 'cpd03737', 'cpd03884', 'cpd04349', 'cpd05158', 'cpd05161', 'cpd05192', 'cpd01356', 'cpd00212', 'cpd00082',
    'cpd13391', 'cpd13392', 'cpd09394', 'cpd00041', 'cpd00129', 'cpd00156', 'cpd00132', 'cpd00107', 'cpd00039', 'cpd20885', 'cpd16654', 'cpd00121',
    'cpd00232', 'cpd00105', 'cpd24451', 'cpd22832', 'cpd11585', 'cpd15584', 'none', 'none', 'cpd37280', 'cpd26351', 'cpd26269', 'cpd25518',
    'cpd24454', 'cpd24453', 'cpd24452', 'cpd24429', 'cpd24428', 'cpd24427', 'cpd24426', 'cpd24425', 'cpd24424', 'cpd24423', 'cpd24420', 'cpd24419',
    'cpd24418', 'cpd24417', 'cpd24416', 'cpd23862', 'cpd23861', 'cpd23859', 'cpd23030', 'cpd16648', 'cpd11603', 'cpd11589',
]
# C_CANDIDATES = ['cpd00276']
# 碳源测试：屏蔽葡萄糖，开放质子交换，打印详细错误
c_results = run_biolog_test(
    candidates=C_CANDIDATES,
    compartment="_e0",
    blocked_exchange="EX_glc_D_e",
    set_h_exchange=True,
    show_errors=True,
)
print("可用碳源数:", sum(1 for v in c_results.values() if v > 1e-6))

0	'cpd00020_e0'
0	None (infeasible).
7.415003498831579
0	None (infeasible).
0	None (infeasible).
0	None (infeasible).
0	'cpd00040_e0'
0	None (infeasible).
0	'cpd00064_e0'
0	'cpd00072_e0'
7.473029452947166
0	'cpd00079_e0'
0	'cpd00080_e0'
0	'cpd00089_e0'
0	None (infeasible).
2.5721003064254746
0	'cpd00106_e0'
0	'cpd00117_e0'
0	'cpd00118_e0'
0	'cpd00122_e0'
0	'cpd00130_e0'
0	'cpd00136_e0'
0	None (infeasible).
0	'cpd00138_e0'
0	None (infeasible).
0	'cpd00141_e0'
0	'cpd00142_e0'
0	None (infeasible).
0	'cpd00155_e0'
7.349853992952969
0	'cpd00158_e0'
0	'cpd00159_e0'
0	None (infeasible).
0	None (infeasible).
0	'cpd00180_e0'
0
0	None (infeasible).
0	'cpd00185_e0'
0	'cpd00208_e0'
0	'cpd00211_e0'
7.374214027370167
0	'cpd00224_e0'
0	'cpd00227_e0'
0	None (infeasible).
0	None (infeasible).
0	None (infeasible).
0	None (infeasible).
0	None (infeasible).
8.073832200931397
0	'cpd00280_e0'
0	'cpd00281_e0'
0	'cpd00306_e0'
0	'cpd00308_e0'
0	'cpd00314_e0'
0	'cpd00320_e0'
0	None (infeasible).
0	'cpd00339_e0'

### <a id="sec-6-3"></a> 6.3 氮源测试（N）

候选氮源位于胞质（`_c0`）。屏蔽 NH₄⁺；`cpd00064` 无对应转运反应，仅测试分泌方向。

In [14]:
# Biolog 氮源候选列表
N_CANDIDATES = [
    'cpd00013', 'cpd00033', 'cpd00064', 'cpd00073', 'cpd00075', 'cpd00092', 'cpd00117', 'cpd00118', 'cpd00128', 'cpd00151', 'cpd00152', 'cpd00161',
    'cpd00162', 'cpd00165', 'cpd00182', 'cpd00184', 'cpd00186', 'cpd00187', 'cpd00207', 'cpd00209', 'cpd00227', 'cpd00246', 'cpd00249', 'cpd00023',
    'cpd00053', 'cpd00274', 'cpd00276', 'cpd00281', 'cpd00300', 'cpd00307', 'cpd00309', 'cpd00311', 'cpd00312', 'cpd00320', 'cpd00322', 'cpd00339',
    'cpd00367', 'cpd00374', 'cpd00378', 'cpd00477', 'cpd00492', 'cpd00054', 'cpd00084', 'cpd00549', 'cpd00550', 'cpd00119', 'cpd00591', 'cpd00065',
    'cpd11809', 'cpd00035', 'cpd00069', 'cpd01092', 'cpd00060', 'cpd01217', 'cpd01258', 'cpd01293', 'cpd01308', 'cpd00066', 'cpd01523', 'cpd01524',
    'cpd01588', 'cpd00051', 'cpd12230', 'cpd02241', 'cpd03161', 'cpd03734', 'cpd03840', 'cpd03963', 'cpd04763', 'cpd00041', 'cpd00129', 'cpd00156',
    'cpd00132', 'cpd00107', 'cpd00039', 'cpd19969', 'cpd15388', 'cpd11593', 'cpd11587', 'cpd11583', 'cpd11580', 'cpd11591', 'cpd11585', 'cpd24431',
    'cpd24432', 'cpd24433', 'cpd26269', 'cpd23860', 'cpd24435', 'cpd24436', 'cpd11584', 'cpd23857', 'cpd23855', 'cpd24418', 'cpd23854',
]

# 氮源测试：屏蔽 NH4+，不开放质子交换，出错仅打印 0
n_results = run_biolog_test(
    candidates=N_CANDIDATES,
    compartment="_c0",
    blocked_exchange="EX_nh4_e",
    no_uptake_exceptions=("cpd00064",),  # 该代谢物仅允许分泌方向
)
print("可用氮源数:", sum(1 for v in n_results.values() if v > 1e-6))

0.04681405214109399
0
0
0
0
0
0
0
0.034333809858294484
0
0
1.7307414974132622
1.2966953687307428
0.047930014124507474
1.434640239719699
0
1.450318190110835
0
0.030369622804015194
0
0.04592169864815768
0
0
1.4503181901108155
0.052550737239004365
0
6.213411986594309
0
0
0
0
0
0
0
0
0
1.2966953687307285
0
0
0
0
1.985618529906065
0.048057319821394925
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0.047767026609439885
0
0
0
2.0303452648503795
0
0
0.05161701773779869
0
0
0
0
1.1159400925475056
1.4503181901108213
0
1.6931049588532818
0
0
0
0
0
0
0
0
0
0
1.7307414974132433
2.0353074351376312
1.693104958853281
0
可用氮源数: 24


### <a id="sec-6-4"></a> 6.4 磷源测试（P）

候选磷源位于胞质（`_c0`）。屏蔽磷酸根（`EX_pi_e`），开放质子交换。

In [15]:
# Biolog 磷源候选列表
P_CANDIDATES = [
    'cpd00009', 'cpd00012', 'cpd00018', 'cpd00046', 'cpd00061', 'cpd00079', 'cpd00080', 'cpd00089', 'cpd00091', 'cpd00126', 'cpd00146', 'cpd00169',
    'cpd00235', 'cpd00284', 'cpd00285', 'cpd00288', 'cpd00298', 'cpd00421', 'cpd00446', 'cpd00457', 'cpd00482', 'cpd00485', 'cpd00696', 'cpd00697',
    'cpd00700', 'cpd00727', 'cpd00738', 'cpd00885', 'cpd00988', 'cpd00989', 'cpd01550', 'cpd01570', 'cpd01571', 'cpd01572', 'cpd01622', 'cpd01665',
    'cpd01908', 'cpd01939', 'cpd01991', 'cpd02233', 'cpd03166', 'cpd03385', 'cpd03454', 'cpd03535', 'cpd03701', 'cpd03702', 'cpd03805', 'cpd03912',
    'cpd08928', 'cpd23708', 'cpd24439', 'cpd24443', 'cpd24445', 'cpd24444', 'cpd24440', 'cpd24438', 'cpd24437',
]
# P_CANDIDATES = ['cpd00738']  # glpk may cannot solve this model, but gurobi can
# 磷源测试：屏蔽磷酸根，开放质子交换
p_results = run_biolog_test(
    candidates=P_CANDIDATES,
    compartment="_c0",
    blocked_exchange="EX_pi_e",
    set_h_exchange=True,
)
print("可用磷源数:", sum(1 for v in p_results.values() if v > 1e-6))

0.04734534623951658
0.04727827591600012
0.047256069633275215
0.04931071678058445
0.04841123186987807
0.048989094548825905
0.04847463585683966
0.04806008317949665
1.2966953687307328
0.04751862560891746
0.06815128771983615
0.0484112318698781
0.048989094548825926
1.6832470830018864
0
0.05356292722989069
0
0.04746126782953559
0.04712262272358615
0.04734534623951668
0.04841123186987815
0.0489890945488259
1.5130574459526278
1.320250802295583
1.4346402397196967
0.04734534623951657
0.04984937338262972
0
1.4346402397197013
0
1.2966953687307385
1.4346402397197011
0
1.296695368730746
0
1.296695368730742
0.0477557955671621
1.2966953687307439
0
0
0
0
0
1.3642442778845614
1.320250802295581
1.3202508022955755
0
1.3619724317106399
1.653763973068161
1.3202508022955757
1.2966953687307414
0
0
0
0
0.04734534623951657
0
可用磷源数: 40


### <a id="sec-6-5"></a> 6.5 硫源测试（S）

候选硫源位于胞质（`_c0`）。屏蔽硫酸根（`EX_so4_e`）。

In [16]:
# Biolog 硫源候选列表
S_CANDIDATES = [
    'cpd00042', 'cpd00048', 'cpd00210', 'cpd00213', 'cpd00268', 'cpd00395', 'cpd00406', 'cpd00424', 'cpd00467', 'cpd00084', 'cpd00587', 'cpd00637',
    'cpd01017', 'cpd01160', 'cpd00060', 'cpd01414', 'cpd01756', 'cpd01914', 'cpd03047', 'cpd03048', 'cpd03776', 'cpd04181', 'cpd05190', 'cpd08023',
    'cpd08093', 'cpd10114', 'cpd24447', 'cpd11591', 'cpd11577', 'none', 'cpd28135', 'cpd24449', 'cpd24448', 'cpd24445', 'cpd24444',
]

# 硫源测试：屏蔽硫酸根，不开放质子交换
s_results = run_biolog_test(
    candidates=S_CANDIDATES,
    compartment="_c0",
    blocked_exchange="EX_so4_e",
)
print("可用硫源数:", sum(1 for v in s_results.values() if v > 1e-6))

0
0.046823647285637655
0
0
0.064878910119569
0
0
0
0
1.300470462657672
0
0
1.5592374224406673
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
0
可用硫源数: 4


## <a id="sec-7"></a> 7. 附录
- FBA 无氧生物量约 0.047；
- 各 Biolog 测试的可用底物数量见第 6 节各测试的运行输出；
- 若更换数据文件，只需修改第 1.2 节的 `EXCEL_FILE` 与 `MODEL_NAME`。